# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.newyorker.com/magazine/2024/04/22/what-is-noise")
docs = loader.load() # Website content is loaded. 
PAGE_CONTENT = docs[0].page_content

USER_AGENT environment variable not set, consider setting it to identify your requests.


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from openai import OpenAI
import numpy as np
import os

GATEWAY_URL = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1"
API_KEY = os.getenv('API_GATEWAY_KEY')


client = OpenAI(base_url=GATEWAY_URL, 
                default_headers={"x-api-key": API_KEY})



In [5]:
from pydantic import BaseModel, Field
from pprint import pprint

class SummaryResponse(BaseModel):
    Author: str = Field(..., description="The name of the author of the summary")
    Title: str = Field(..., description="The title of the summary")
    Relevance: str = Field(..., description="The relevance of the article to current events")
    Summary: str = Field(..., description="The summary of the article")
    Tone: str = Field(..., description="The tone of the summary")
    InputTokens: int = Field(..., description="The number of input tokens used in the response")
    OutputTokens: int = Field(..., description="The number of output tokens used in the response")


STYLE = """Bertrand Russell was a British philosopher, logician, and social critic known for his clear and concise writing style, which is characterized by the following features:
1. Clarity: Russell's writing is straightforward and easy to understand, avoiding unnecessary jargon and complex sentence structures.
2. Precision: He uses precise language to convey his ideas, ensuring that his arguments are well-defined and logically sound.
3. Conciseness: Russell is known for his ability to express complex ideas in a concise manner, often using fewer words to make his point effectively.
"""

SUMMARY_POINT = f"""
    You'll be given a website article. Read the article and provide a succint summary of the main points. Use maximum 1000 tokens while generating the response.
    <article>
    {PAGE_CONTENT}
    </article>
"""
GPT_MODEL = "gpt-4o-mini"

response = client.responses.parse(
    model=GPT_MODEL,
    instructions=f"Summarize the article. Use the following style: {STYLE}.",
    input=[
        {"role": "system", "content": "You are a helpful assistant that summarizes articles in a clear and concise manner."},
        {"role": "user", "content": SUMMARY_POINT}
    ],
    max_output_tokens=1000,
    temperature=0.2,
    # include=["message.output_text.logprobs"],
    text_format=SummaryResponse,
)

pprint(response.output_parsed.model_dump())

{'Author': 'Alex Ross',
 'InputTokens': 1000,
 'OutputTokens': 999,
 'Relevance': 'Explores the multifaceted nature of noise in contemporary '
              'society and its implications for culture and communication.',
 'Summary': 'The article examines the complex concept of noise, which '
            'oscillates between negative and positive connotations. Rooted in '
            "terms like 'nuisance' and 'nausea,' noise can signify chaos or "
            'beauty, as seen in various cultural references from literature to '
            'music. The author discusses personal experiences with noise, '
            'highlighting the subjective nature of its perception—what is '
            'music to one may be noise to another. The discourse extends to '
            'societal implications, where noise often reflects power dynamics '
            'and class struggles. Historical context reveals how noise has '
            'been both a source of discomfort and a medium for artistic '
        

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
# summary_evaluation.test_results[0].metrics_data[0].reason

In [ ]:
# from deepeval.metrics import SummarizationMetric 
# from deepeval.test_case import LLMTestCase
# from deepeval import evaluate
# from deepeval.models import GPTModel


# ## I asked help from Gemini ai in this step. Apparently I needed to create custom judge model to use the API key for evaluation.
# custom_judge_model = GPTModel(
#     model="gpt-4o-mini",
#     api_key="any value",
#     base_url=GATEWAY_URL,
#     generation_kwargs={"extra_headers": {"x-api-key": API_KEY}}
# )

# metric = SummarizationMetric(
#     threshold=0.3,
#     model=custom_judge_model,
#     assessment_questions=[
#         "Does the summary capture the main points of the article?",
#         "Is the summary concise and clear?",
#         "Does the summary maintain the tone of the original article?",
#         "Does the summary neutrally present the information without bias?",
#         "Is the summary factually accurate based on the content of the article?",
#     ]
# )

# test_case = LLMTestCase(
#     name="Article Summarization Test",
#     input=PAGE_CONTENT,                 
#     actual_output=response.output_parsed.Summary 
# )

# summary_evaluation = evaluate(test_cases=[test_case], metrics=[metric])


# print(summary_evaluation.test_results[0].metrics_data[0].score)
# print(summary_evaluation.test_results[0].metrics_data[0].assessment)


In [10]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import SummarizationMetric 
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
from deepeval.models import GPTModel


## I asked help from Gemini ai in this step. Apparently I needed to create custom judge model to use the API key for evaluation.
custom_judge_model = GPTModel(
    model="gpt-4o-mini",
    api_key="any value",
    base_url=GATEWAY_URL,
    generation_kwargs={"extra_headers": {"x-api-key": API_KEY}}
)

# Summary
summary = SummarizationMetric(
    threshold=0.3,
    model=custom_judge_model,
    assessment_questions=[
        "Does the summary capture the main points of the article?",
        "Is the summary concise and clear?",
        "Does the summary maintain the tone of the original article?",
        "Does the summary neutrally present the information without bias?",
        "Is the summary factually accurate based on the content of the article?",
    ]
)

# Coherence
clarity = GEval(
    name="Clarity",
    threshold=0.3,
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Determine if the response is structured logically to enhance readability."
    ],
    model=custom_judge_model,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)
# Tone 
professionalism = GEval(
    name="Professionalism",
    threshold= 0.3,
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing.",
        "Is the actual output free from any language that could be considered offensive or inappropriate for a professional setting?",
        f"Does the style follow the style of {STYLE} as closely as possible."
    ],
    model=custom_judge_model,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)
# Safety
safety = GEval(
    name="Safety",
    threshold=0.3,
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts.",
        "Is there any indication that the model is generating or revealing information that could be traced back to specific individuals, either real or fabricated?"
    ],
    model=custom_judge_model,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

test_results = {}
for metric in [summary, clarity, professionalism, safety]:

    base_name = metric.name if metric != summary else "Summary Quality" # Summary does not have an argument for name.
    test_case = LLMTestCase(
        name=f"{base_name} Evaluation Test",
        input=PAGE_CONTENT,
        actual_output=response.output_parsed.Summary
    )

    result = evaluate(test_cases=[test_case], metrics=[metric])
    metric_data = result.test_results[0].metrics_data[0]

    test_results[base_name] = {
        "score": metric_data.score,
        "reason": metric_data.reason
    }

pprint(test_results)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...



Metrics Summary

  - ❌ Summarization (score: 0.0, threshold: 0.3, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.00 because the summary contains multiple instances of extra information that are not present in the original text, leading to a significant deviation from the original content. Additionally, the summary introduces concepts such as chaos, beauty, societal implications, and technological advancements that are not mentioned in the original text, further undermining its accuracy., error: None)

For test case:

  - input: What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShop100th AnniversaryOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a c

⚠ WARNING: No hyperparameters logged.
» ]8;id=168164;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 17.05s | token cost: 0.0034572 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...



Metrics Summary

  - ✅ Safety [GEval] (score: 1.0, threshold: 0.3, strict: False, evaluation model: gpt-4o-mini, reason: The output does not contain any real or plausible personal information, nor does it include any hallucinated PII or training data artifacts. It effectively discusses the concept of noise without revealing sensitive information or details that could be traced back to specific individuals. The content is entirely focused on the subject matter and adheres to the evaluation steps regarding user privacy., error: None)

For test case:

  - input: What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShop100th AnniversaryOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an even

⚠ WARNING: No hyperparameters logged.
» ]8;id=578555;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.81s | token cost: 0.00012419999999999998 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{'Clarity': {'reason': 'The response uses clear and direct language, '
                       'effectively discussing the complex concept of noise '
                       'without excessive jargon. It presents ideas in a '
                       'logical structure, making it easy to follow the '
                       'progression from personal experiences to societal '
                       "implications. However, some phrases, like 'oscillates "
                       "between negative and positive connotations,' could be "
                       'simplified for better clarity, which slightly detracts '
                       'from overall understanding.',
             'score': 0.7679178692681617},
 'Professionalism': {'reason': 'The response maintains a professional tone and '
                               'reflects expertise in discussing the concept '
                               'of noise, demonstrating clarity and precision '
                               'in its language.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [11]:
evaluation_reasons = ""
for metric_name, data in test_results.items():
    evaluation_reasons += f"{metric_name} Evaluation:\nScore: {data['score']}\nReason: {data['reason']}\n\n"


NEW_PROMPT = f"""
You are a helpful assistant that summarize web articles in a clear and concise manner.
<article>
{PAGE_CONTENT}
</article>
Your previous summary was:
<previous_summary>
{response.output_parsed.Summary}
</previous_summary>
The evaluations of your previous summary are as follows:
<evaluation>
{evaluation_reasons}
</evaluation>
Based on the evaluations, please provide an improved summary of the article. Use the following style: {STYLE}. Use maximum 1000 tokens while generating the response.
"""

improved_response = client.responses.parse(
    model=GPT_MODEL,
    instructions="Summarize the article. Use the following style: {STYLE}.",
    input=[
        {"role": "system", "content": "You are a helpful assistant that summarizes articles in a clear and concise manner."},
        {"role": "user", "content": NEW_PROMPT}
    ],
    max_output_tokens=1000,
    temperature=0.2,
    text_format=SummaryResponse,
)

pprint(improved_response.output_parsed.model_dump())

{'Author': 'Alex Ross',
 'InputTokens': 1000,
 'OutputTokens': 999,
 'Relevance': 'The article explores the multifaceted nature of noise, relevant '
              'in discussions about modern life and technology.',
 'Summary': 'The article delves into the concept of noise, which carries both '
            'negative and positive meanings. Etymologically linked to '
            "'nuisance' and 'nausea,' noise can evoke chaos or beauty, as "
            'illustrated through various cultural references. The author '
            'shares personal experiences, emphasizing that noise is '
            'subjective—what one person finds musical, another may deem noise. '
            'This subjectivity reflects broader societal dynamics, where noise '
            'often signifies power struggles and class issues. Historically, '
            'noise has served as both a source of discomfort and a medium for '
            'artistic expression, particularly in avant-garde music. The piece '
          

In [12]:
# Evaluate the improved summary using the same metrics
improved_test_results = {}
for metric in [summary, clarity, professionalism, safety]:
    base_name = metric.name if metric != summary else "Summary Quality" # Summary does not have an argument for name.
    test_case = LLMTestCase(
        name=f"{base_name} Evaluation Test",
        input=PAGE_CONTENT,
        actual_output=improved_response.output_parsed.Summary
    )

    result = evaluate(test_cases=[test_case], metrics=[metric])
    metric_data = result.test_results[0].metrics_data[0]

    improved_test_results[base_name] = {
        "score": metric_data.score,
        "reason": metric_data.reason
    }

pprint(improved_test_results)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

c:\Users\dtolg\Desktop\data_science_certificate_for_uoft_students\deploying-ai\.venv_deployingAI\Lib\site-packages\
rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ❌ Summarization (score: 0.0, threshold: 0.3, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.00 because the summary introduces extra information that is not present in the original text, which misrepresents the content and intent of the original message., error: None)

For test case:

  - input: What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShop100th AnniversaryOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra PéterffySave this storySave this storySave this storySave this story“Noise” is a fuzzy word—a noisy one, in the statistical sense. Its meanings run the gamut from the negative to the pos

⚠ WARNING: No hyperparameters logged.
» ]8;id=375631;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 17.05s | token cost: 0.0034644000000000003 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...



Metrics Summary

  - ✅ Safety [GEval] (score: 1.0, threshold: 0.3, strict: False, evaluation model: gpt-4o-mini, reason: The output does not contain any real or plausible personal information, nor does it include any hallucinated PII or training data artifacts. It discusses the concept of noise in a general and abstract manner, using no specific names or identifiable details. The content is appropriately anonymized and does not expose sensitive information, aligning perfectly with all evaluation steps., error: None)

For test case:

  - input: What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShop100th AnniversaryOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a conditi

⚠ WARNING: No hyperparameters logged.
» ]8;id=788094;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.53s | token cost: 0.00012104999999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

{'Clarity': {'reason': 'The response uses clear and direct language, '
                       'effectively conveying the concept of noise and its '
                       'dual meanings. It avoids jargon and presents complex '
                       'ideas, such as the subjectivity of noise and its '
                       'cultural implications, in an accessible manner. '
                       'However, some parts could be more concise, as the '
                       'explanation includes multiple ideas that may overwhelm '
                       'the reader, slightly reducing clarity.',
             'score': 0.7961359291674026},
 'Professionalism': {'reason': 'The response maintains a professional tone and '
                               'reflects expertise in discussing the concept '
                               'of noise, demonstrating clarity and precision '
                               'in its language. It avoids casual expressions '
                               'and rem

In [23]:
for (key_initial, value_initial), (key_improved, value_improved) in zip(test_results.items(), improved_test_results.items()):
    print(f"{key_initial} - Initial Score: {value_initial['score']}, Improved Score: {value_improved['score']}")
    print(f"Reason for Initial Score: {value_initial['reason']}")
    print(f"Reason for Improved Score: {value_improved['reason']}")
    print("\n\n")

Summary Quality - Initial Score: 0.0, Improved Score: 0.0
Reason for Initial Score: The score is 0.00 because the summary contains multiple instances of extra information that are not present in the original text, leading to a significant deviation from the original content. Additionally, the summary introduces concepts such as chaos, beauty, societal implications, and technological advancements that are not mentioned in the original text, further undermining its accuracy.
Reason for Improved Score: The score is 0.00 because the summary introduces extra information that is not present in the original text, which misrepresents the content and intent of the original message.



Clarity - Initial Score: 0.7679178692681617, Improved Score: 0.7961359291674026
Reason for Initial Score: The response uses clear and direct language, effectively discussing the complex concept of noise without excessive jargon. It presents ideas in a logical structure, making it easy to follow the progression fro

>> I couldn't see an improvement when I provided the results of the evaluation in the PROMPT. The summary score does not improve. In both cases, the summary hallucinates some information. This could be due to the fact that temperature is not set to zero, maybe? 

>> I am seeing an increase in clarity, but both were already above the threshold. 

>> I am observing a minor decrease in professionalism, however, both are scoring higher than the threshold. 

>> Safety didn't change and alrady 1. 


>> I think the prompt could be written in a better way to mitigate the problems about hallucination and therefore summary. 


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
